# NB70: Multi-Cloud Replication

Kafka -> Spark -> MinIO/Mongo/ES

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip uninstall -y numpy
!pip install -q "numpy<2.0.0" pandas scipy
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **MongoDB**
- **Elasticsearch**

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start MongoDB
!wget -qO - https://www.mongodb.org/static/pgp/server-6.0.asc | apt-key add -
!echo "deb [ arch=amd64,arm64 ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/6.0 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-6.0.list
!apt-get update -qq > /dev/null
!apt-get install -y mongodb-org -qq > /dev/null
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --bind_ip 127.0.0.1
# Start Elasticsearch
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.10.2-linux-x86_64.tar.gz
!tar -xzf elasticsearch-7.10.2-linux-x86_64.tar.gz
!chown -R daemon:daemon elasticsearch-7.10.2
!sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m" ./elasticsearch-7.10.2/bin/elasticsearch -d > es.log 2>&1 &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9200) # Elasticsearch
wait_for_port(27017) # MongoDB


In [ ]:
# Start MinIO on Custom Port 9010 (Shared with NB62/NB63/NB68 convention)
!wget -q https://dl.min.io/server/minio/release/linux-amd64/minio
!chmod +x minio
!mkdir -p /content/minio_data_nb70
!MINIO_ROOT_USER=minioadmin MINIO_ROOT_PASSWORD=minioadmin ./minio server /content/minio_data_nb70 --address ":9010" --console-address ":9011" &> minio_9010.log &

# Wait for MinIO 9010
import time, socket, os
print('Waiting for MinIO on 9010...')
start = time.time()
while True:
    try:
        with socket.create_connection(('localhost', 9010), timeout=1): break
    except (OSError, ConnectionRefusedError):
        if time.time() - start > 120:
             if os.path.exists('minio_9010.log'): print(open('minio_9010.log').read())
             raise Exception('MinIO 9010 Failed')
        time.sleep(1)
print('MinIO 9010 Ready!')

## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer

Sends raw data to be replicated.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Replication Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 100 items...")
for i in range(100):
    producer.send('input-topic', f'data_{i}'.encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Replication Job

Replicates each message to Mongo and Elasticsearch simultaneously.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import json
from pymongo import MongoClient
from elasticsearch import Elasticsearch

spark = SparkSession.builder.appName("MultiCloud").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    if not rows: return
    mongo = MongoClient()
    es = Elasticsearch(['http://localhost:9200'])
    for row in rows:
        val = row.value.decode('utf-8')
        # Replicate
        mongo.cloud.replica.insert_one({'val': val})
        es.index(index='replica', body={'val': val})
    print(f"Batch {epoch_id} replicated to Mongo & ES")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check Replication counts.

In [ ]:
from pymongo import MongoClient
from elasticsearch import Elasticsearch

m = MongoClient()
print(f"Mongo Count: {m.cloud.replica.count_documents({})}")

es = Elasticsearch(['http://localhost:9200'])
time.sleep(2)
print(f"ES Count: {es.count(index='replica')['count']}")